# Quickstart: Detecting Structural Breaks via the Adaptive Fused Lasso

This notebook shows a minimal end-to-end example using the package:

1. simulate one panel dataset with a single structural break,
2. choose the tuning parameter with the information criterion,
3. estimate the break structure,
4. plot the IC curve,
5. plot the true and estimated coefficient paths.

This is a small demo meant to help users test that the code runs and produces sensible plots.

In [ ]:
%pip install -q cvxpy clarabel numpy scipy matplotlib tabulate

import numpy as np
from src.dgp import DATA3
from src.estimator import Optimize
from src.ic import information_criterion
from src.utils import plot_ic_curve, plot_beta_path

In [ ]:
np.random.seed(123)

# Dimensions
n, T, p, m, r = 25, 5, 4, 1, 5

# Generate one-break panel dataset
data = DATA3(r=r, m=m, T=T, n=n, p=p, phi=0.8, phi_1=0.4, pi=0.4)
X, y, beta_true, u, eps, F, y_tilde, u_tilde, X_mean, X_tilde = data.DGP1()

print("Shapes:")
print("X:", X.shape)
print("y:", y.shape)
print("beta_true:", beta_true.shape)
print("X_tilde:", X_tilde.shape)
print("y_tilde:", y_tilde.shape)

In [ ]:
lam_grid = np.logspace(-3, 3, 50)

IC_vec, m_breaks, IC_min, lam_idx, lam_star, m_star = information_criterion(
    lam_grid, y_tilde, X_tilde, p, T, n
)

print("Selected lambda:", lam_star)
print("Estimated number of breaks at lambda*:", int(m_star))
print("Minimum IC:", IC_min)

In [ ]:
opt = Optimize(p, T, n)

# First-stage OLS
b_ols, status_ols, value_ols = opt.OLS(X_tilde, y_tilde)

# Fused lasso estimate at selected lambda
b_hat, m_hat, status_fgls, value_fgls = opt.FGLS(X_tilde, y_tilde, b_ols, lam_star)

print("OLS status:", status_ols)
print("FGLS status:", status_fgls)
print("True number of breaks:", m)
print("Estimated number of breaks:", m_hat)

In [ ]:
opt = Optimize(p, T, n)

# First-stage OLS
b_ols, status_ols, value_ols = opt.OLS(X_tilde, y_tilde)

# Fused lasso estimate at selected lambda
b_hat, m_hat, status_fgls, value_fgls = opt.FGLS(X_tilde, y_tilde, b_ols, lam_star)

print("OLS status:", status_ols)
print("FGLS status:", status_fgls)
print("True number of breaks:", m)
print("Estimated number of breaks:", m_hat)

In [ ]:
fig_ic = plot_ic_curve(lam_grid, IC_vec, m_breaks, lam_star)

## Interpretation

If the code is working well, the selected model should detect one break, and the estimated coefficient paths should track the true regime shift reasonably closely.

Because the data are simulated, exact estimates vary across random seeds, but the overall pattern should be clear.